# `cuda_core` smoke test — `cuda.core` (NVIDIA cuda-python)

The CUDA-touching example (cell 6) is **GPU-conditional**: it asks the CUDA runtime how many
devices are visible and prints `SKIP (no GPU)` when there are none (or when no NVIDIA driver is
loaded to reach one), so the notebook passes both with and without a GPU. Any other CUDA error
fails the cell.

In [ ]:
import pkgutil, importlib
import cuda.core as _pkg_under_test

# Excluded submodule, with its reason:
_DENYLIST = frozenset({
    # Optional PyTorch AOTInductor bridge: its `aoti_torch_*` symbols live in libtorch
    # and resolve only through the bridge's own loader, so a cold import from this walk
    # fails -- not a defect.
    "cuda.core._tensor_bridge",
})

# Walk with the public prefix "cuda.core.": NVIDIA's wheel re-exports a per-CUDA-major
# submodule from `import cuda.core` (for example `cuda.core.cu13`), which also changes
# `cuda.core.__name__`.
failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix="cuda.core."):
    if module_info.name in _DENYLIST:
        continue
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of cuda.core failed to import")
print("OK — all submodules of cuda.core imported cleanly")

In [ ]:
import cuda.core, cuda.bindings

print("cuda.core   :", cuda.core.__version__)
print("cuda.bindings:", cuda.bindings.__version__)
assert isinstance(cuda.core.__version__, str) and cuda.core.__version__[0].isdigit()
assert isinstance(cuda.bindings.__version__, str)

# `system.*` initializes and queries NVML — present wherever the NVIDIA driver is loaded; the
# marker above skips this cell where there is none, so here the query must succeed. (NVML ignores CUDA_VISIBLE_DEVICES,
# so `get_num_devices` reports the physical count, not the CUDA-visible one.)
from cuda.core import system
try:
    print("driver (user-mode):", system.get_user_mode_driver_version())
    print("nvml version      :", system.get_nvml_version())
    print("nvml sees devices :", system.get_num_devices())
except Exception as e:
    print(f"NVML driver introspection unavailable (no driver loaded): {type(e).__name__}")
    raise

In [ ]:
# NVRTC runtime compilation (cuda.core.Program wraps NVRTC) — needs NO GPU, exercises the
# real JIT compiler runtime. Compile a SAXPY kernel to PTX and inspect the emitted code.
from cuda.core import Program, ProgramOptions

src = (
    'extern "C" __global__ void axpy(float a, const float* x, const float* y, float* out, int n) {\n'
    '    int i = blockIdx.x * blockDim.x + threadIdx.x;\n'
    '    if (i < n) out[i] = a * x[i] + y[i];\n'
    '}\n'
)
obj = Program(src, code_type="c++", options=ProgramOptions(arch="sm_90a", std="c++17")).compile("ptx")
ptx = bytes(obj.code).decode("utf-8", "replace")
assert obj.code_type == "ptx" and ".visible .entry axpy" in ptx, ptx[:200]
print(f"NVRTC -> PTX OK: {len(ptx)} chars, kernel 'axpy' present (code_type={obj.code_type})")

In [ ]:
# Invariant / error-class probe: a well-formed kernel compiles; a malformed one raises a
# diagnostic. Exercises NVRTC's error path (the class most likely to regress silently).
from cuda.core import Program, ProgramOptions
# cuda.core 1.2.0 exports no exception class publicly; Program.compile raises this one on a compile error.
from cuda.core._utils.cuda_utils import NVRTCError

opts = ProgramOptions(arch="sm_90a")
Program('extern "C" __global__ void ok() {}', code_type="c++", options=opts).compile("ptx")

try:
    Program('extern "C" __global__ void bad() { undeclared_fn(); }',
            code_type="c++", options=opts).compile("ptx")
except NVRTCError as e:
    raised = e
else:
    raise AssertionError("malformed kernel unexpectedly compiled")
assert "NVRTC_ERROR_COMPILATION" in str(raised), raised
assert 'identifier "undeclared_fn" is undefined' in str(raised), raised
print(f"good kernel compiled; malformed kernel raised {type(raised).__name__} with a diagnostic: OK")

In [ ]:
# nvJitLink device-link (cuda.core.Linker): compile two device translation units to LTO-IR
# and link them into one cubin. CPU-safe — linking needs no GPU.
from cuda.core import Program, ProgramOptions, Linker, LinkerOptions

lo = ProgramOptions(arch="sm_90a", relocatable_device_code=True, link_time_optimization=True)
dev_fn = 'extern "C" __device__ float sq(float x) { return x * x; }'
kernel = ('extern "C" __device__ float sq(float);\n'
          'extern "C" __global__ void apply(const float* x, float* o, int n) {\n'
          '    int i = threadIdx.x; if (i < n) o[i] = sq(x[i]); }')
a = Program(dev_fn, code_type="c++", options=lo).compile("ltoir")
b = Program(kernel, code_type="c++", options=lo).compile("ltoir")
linked = Linker(a, b, options=LinkerOptions(arch="sm_90a", link_time_optimization=True)).link("cubin")
assert linked.code_type == "cubin" and len(bytes(linked.code)) > 0
print(f"nvJitLink LTO link OK: 2 LTO-IR TUs -> {linked.code_type}, {len(bytes(linked.code))} bytes")

In [ ]:
import numpy as np
from cuda.bindings import runtime as rt
from cuda.core import Program, ProgramOptions, Device, LaunchConfig, launch, LegacyPinnedMemoryResource

# Skip only when the CUDA runtime sees no device (or no driver to reach one); any other
# failure (device query, context, launch) raises.
err, count = rt.cudaGetDeviceCount()
_no_device = (rt.cudaError_t.cudaErrorNoDevice, rt.cudaError_t.cudaErrorInsufficientDriver)
if err in _no_device or (err == rt.cudaError_t.cudaSuccess and count == 0):
    print(f"SKIP (no GPU): cudaGetDeviceCount -> {err}, count={count}")
else:
    assert err == rt.cudaError_t.cudaSuccess, err
    dev = Device(0); dev.set_current()
    cc = dev.compute_capability
    src = ('extern "C" __global__ void axpy(float a, const float* x, const float* y, float* o, int n)'
           '{int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) o[i]=a*x[i]+y[i];}')
    ker = Program(src, code_type="c++", options=ProgramOptions(arch=f"sm_{cc.major}{cc.minor}")).compile("cubin").get_kernel("axpy")
    s = dev.create_stream(); pin = LegacyPinnedMemoryResource(); n = 256
    def host(vals):
        b = pin.allocate(n * 4, stream=s); np.from_dlpack(b).view(np.float32)[:] = vals; return b
    hx, hy, ho = host(np.ones(n, np.float32)), host(np.full(n, 3.0, np.float32)), host(np.zeros(n, np.float32))
    dx, dy, do = (dev.allocate(n * 4, stream=s) for _ in range(3))
    dx.copy_from(hx, stream=s); dy.copy_from(hy, stream=s)
    launch(s, LaunchConfig(grid=(n + 63) // 64, block=64), ker, np.float32(2.0), dx, dy, do, np.int32(n))
    ho.copy_from(do, stream=s); s.sync()
    assert np.allclose(np.from_dlpack(ho).view(np.float32), 5.0)
    print(f"cuda.core launch OK: axpy(2*1+3) on {n} elems == 5.0 ({dev.name}, sm_{cc.major}{cc.minor})")